# Programowanie w języku Python

## Wykład 11 — OOP II: relacje między klasami i Python Data Model

### dr inż. Waldemar Bauer

## Cel wykładu

Po dzisiejszych zajęciach powinieneś:

- rozumieć rolę `__repr__` i `__str__`,
- wiedzieć, jak klasa może definiować znaczenie operatorów,
- poprawnie implementować `__eq__`,
- rozumieć podstawy dziedziczenia i `super()`,
- rozumieć polimorfizm i duck typing,
- odróżniać dziedziczenie od kompozycji,
- znać zastosowanie `@classmethod` i `@staticmethod`,
- umieć korzystać z `@dataclass`,
- rozumieć uproszczony mechanizm wyszukiwania atrybutów,
- wiedzieć, czym są deskryptory i metaklasy na poziomie koncepcyjnym.

## Powtórzenie z OOP I

Na poprzednim wykładzie zbudowaliśmy obiekt od środka:

```text
OBIEKT

STAN
↓
atrybuty

ZACHOWANIE
↓
metody
```

Dzisiaj zobaczymy:

> jak obiekt współpracuje z samym językiem Python i jak klasy mogą pozostawać ze sobą w relacjach.

## Punkt wyjścia

Mamy klasę:

```python
class TemperatureSensor:

    def __init__(self, name, value=None):
        self.name = name
        self.value = value
```

i obiekt:

```python
sensor = TemperatureSensor("sensor_01", 21.7)
```

Co zobaczymy po:

```python
print(sensor)
```?

In [ ]:
class TemperatureSensor:

    def __init__(self, name, value=None):
        self.name = name
        self.value = value

sensor = TemperatureSensor("sensor_01", 21.7)

print(sensor)

## Domyślna reprezentacja obiektu

Python potrafi wyświetlić obiekt nawet wtedy, gdy nie zdefiniowaliśmy własnej reprezentacji.

Ale wynik:

```text
<__main__.TemperatureSensor object at 0x...>
```

jest mało użyteczny.

Możemy nauczyć klasę, jak ma reprezentować swoje obiekty.

## `__repr__`

`__repr__` powinien dawać reprezentację przydatną przede wszystkim dla programisty i diagnostyki.

In [ ]:
class TemperatureSensor:

    def __init__(self, name, value=None):
        self.name = name
        self.value = value

    def __repr__(self):
        return (
            f"TemperatureSensor("
            f"name={self.name!r}, "
            f"value={self.value!r})"
        )

sensor = TemperatureSensor("sensor_01", 21.7)

print(repr(sensor))
print(sensor)

## Dlaczego używamy `!r`?

W f-stringu:

```python
f"{value!r}"
```

Python korzysta z `repr(value)`.

Dla napisu:

```python
"sensor_01"
```

otrzymamy reprezentację z cudzysłowami:

```text
'sensor_01'
```

To często lepiej pokazuje rzeczywistą strukturę danych.

## `__str__`

`__str__` służy do bardziej czytelnej reprezentacji dla użytkownika.

In [ ]:
class TemperatureSensor:

    def __init__(self, name, value=None):
        self.name = name
        self.value = value

    def __repr__(self):
        return (
            f"TemperatureSensor("
            f"name={self.name!r}, "
            f"value={self.value!r})"
        )

    def __str__(self):
        return f"{self.name}: {self.value} °C"

sensor = TemperatureSensor("sensor_01", 21.7)

print(repr(sensor))
print(str(sensor))
print(sensor)

## `__repr__` a `__str__`

```text
__repr__
↓
reprezentacja techniczna
dla programisty / diagnostyki

__str__
↓
reprezentacja czytelna
dla użytkownika
```

To rozwija funkcje `repr()` i `str()`, które znamy już z wcześniejszych wykładów.

## Operator `==`

Przypomnijmy problem z OOP I:

In [ ]:
sensor_1 = TemperatureSensor("sensor_01", 21.7)
sensor_2 = TemperatureSensor("sensor_01", 21.7)

print(sensor_1 is sensor_2)
print(sensor_1 == sensor_2)

## Co oznacza równość?

Dla naszych obiektów musimy zdecydować:

> kiedy dwa obiekty reprezentują tę samą wartość logiczną?

Na przykład:

```text
ten sam name
i
ta sama value
```

Możemy zdefiniować to przez `__eq__`.

In [ ]:
class TemperatureSensor:

    def __init__(self, name, value=None):
        self.name = name
        self.value = value

    def __eq__(self, other):
        if not isinstance(other, TemperatureSensor):
            return NotImplemented

        return (
            self.name == other.name
            and self.value == other.value
        )

sensor_1 = TemperatureSensor("sensor_01", 21.7)
sensor_2 = TemperatureSensor("sensor_01", 21.7)

print(sensor_1 == sensor_2)

## Dlaczego `NotImplemented`, a nie `False`?

Jeżeli drugi obiekt ma inny typ:

```python
sensor == 21.7
```

to nasza metoda nie musi wiedzieć, jak takie porównanie interpretować.

Zwracamy:

```python
NotImplemented
```

czyli:

> ta implementacja nie obsługuje takiego porównania.

## Operatory są częścią interfejsu klasy

Wyrażenie:

```python
a == b
```

może prowadzić do wywołania:

```python
a.__eq__(b)
```

Podobnie inne operatory mają odpowiadające im metody specjalne.

Nie oznacza to jednak, że każda klasa powinna przeciążać wszystkie operatory.

## Przykład: `__lt__`

Dla pomiarów naturalne może być porównanie wartości:

In [ ]:
class Measurement:

    def __init__(self, value):
        self.value = value

    def __lt__(self, other):
        if not isinstance(other, Measurement):
            return NotImplemented

        return self.value < other.value

measurements = [
    Measurement(23.4),
    Measurement(19.1),
    Measurement(21.7),
]

measurements.sort()

print([m.value for m in measurements])

## Nie przeciążamy operatora bez sensu

Przeciążenie powinno odpowiadać naturalnemu znaczeniu operatora.

Na przykład:

```python
sensor_1 + sensor_2
```

nie ma oczywistego znaczenia.

Jeżeli operator wymaga długiego wyjaśnienia, zwykła metoda może być czytelniejsza.

## Dziedziczenie — skąd potrzeba?

Załóżmy, że mamy:

```text
TemperatureSensor
PressureSensor
HumiditySensor
```

Każdy potrzebuje:

```text
name
value
update()
```

Powielanie tego samego kodu sugeruje możliwość wydzielenia wspólnej klasy.

In [ ]:
class Sensor:

    def __init__(self, name):
        self.name = name
        self.value = None

    def update(self, value):
        self.value = value

## Klasa pochodna

```python
class TemperatureSensor(Sensor):
    pass
```

oznacza:

> `TemperatureSensor` dziedziczy po `Sensor`.

Obiekt klasy pochodnej może korzystać z zachowania zdefiniowanego w klasie bazowej.

In [ ]:
class TemperatureSensor(Sensor):
    pass

sensor = TemperatureSensor("temperature_01")
sensor.update(21.7)

print(sensor.name)
print(sensor.value)

## Relacja IS-A

Dziedziczenie często pasuje do relacji:

```text
TemperatureSensor IS-A Sensor
```

czyli:

> czujnik temperatury jest szczególnym przypadkiem czujnika.

To nie jest jednak automatyczna reguła projektowa.

## Rozszerzanie klasy pochodnej

Klasa pochodna może mieć dodatkowy stan:

In [ ]:
class TemperatureSensor(Sensor):

    def __init__(self, name, unit="C"):
        super().__init__(name)
        self.unit = unit

sensor = TemperatureSensor("temperature_01")

print(sensor.name)
print(sensor.unit)
print(sensor.value)

## `super()`

`super()` pozwala korzystać z implementacji wynikającej z hierarchii dziedziczenia.

W przykładzie:

```python
super().__init__(name)
```

uruchamiamy odpowiednią implementację `__init__` wyżej w hierarchii.

Nie warto upraszczać `super()` do „wywołaj rodzica” — przy bardziej złożonym dziedziczeniu działa zgodnie z MRO.

## Nadpisywanie metody

Klasa pochodna może zdefiniować metodę o tej samej nazwie:

In [ ]:
class Sensor:

    def __init__(self, name):
        self.name = name

    def describe(self):
        return f"Sensor {self.name}"


class TemperatureSensor(Sensor):

    def describe(self):
        return f"Temperature sensor {self.name}"


base_sensor = Sensor("S1")
temperature_sensor = TemperatureSensor("T1")

print(base_sensor.describe())
print(temperature_sensor.describe())

## Polimorfizm

Różne klasy mogą odpowiadać na to samo wywołanie w różny sposób.

Przykład:

In [ ]:
class PressureSensor(Sensor):

    def describe(self):
        return f"Pressure sensor {self.name}"


class HumiditySensor(Sensor):

    def describe(self):
        return f"Humidity sensor {self.name}"


sensors = [
    TemperatureSensor("T1"),
    PressureSensor("P1"),
    HumiditySensor("H1"),
]

for sensor in sensors:
    print(sensor.describe())

## Co jest ważne w polimorfizmie?

Kod:

```python
for sensor in sensors:
    print(sensor.describe())
```

nie musi sprawdzać:

```python
if isinstance(sensor, TemperatureSensor):
    ...
elif isinstance(sensor, PressureSensor):
    ...
```

Każdy obiekt sam dostarcza odpowiednie zachowanie.

## Duck typing

Python często idzie jeszcze dalej.

Funkcja może korzystać z obiektu, jeżeli ten oferuje potrzebne zachowanie:

```python
def show_description(obj):
    print(obj.describe())
```

Nie musi pytać o konkretną klasę.

In [ ]:
def show_description(obj):
    print(obj.describe())


class VirtualSensor:

    def describe(self):
        return "Virtual sensor"


show_description(TemperatureSensor("T1"))
show_description(VirtualSensor())

## Zachowanie ważniejsze niż formalna hierarchia

W duck typing istotne jest:

```text
czy obiekt potrafi wykonać potrzebną operację?
```

a nie wyłącznie:

```text
czy dziedziczy po konkretnej klasie?
```

To bardzo charakterystyczny element stylu programowania w Pythonie.

## Dziedziczenie nie zawsze jest najlepszym rozwiązaniem

Załóżmy, że czujnik zapisuje komunikaty.

Czy powinien dziedziczyć po klasie `Logger`?

```text
Sensor IS-A Logger?
```

Raczej nie.

Bardziej naturalne:

```text
Sensor HAS-A Logger
```

## Kompozycja

```python
class Sensor:

    def __init__(self, name, logger):
        self.name = name
        self.logger = logger
```

Obiekt `Sensor` **posiada** inny obiekt i korzysta z jego usług.

In [ ]:
class ConsoleLogger:

    def log(self, message):
        print(f"[LOG] {message}")


class SensorWithLogger:

    def __init__(self, name, logger):
        self.name = name
        self.logger = logger
        self.value = None

    def update(self, value):
        self.value = value
        self.logger.log(
            f"{self.name} updated to {value}"
        )


logger = ConsoleLogger()
sensor = SensorWithLogger("sensor_01", logger)

sensor.update(21.7)

## Dziedziczenie czy kompozycja?

Przydatna heurystyka:

```text
IS-A
↓
rozważ dziedziczenie

HAS-A
↓
rozważ kompozycję
```

To tylko wskazówka, ale pomaga uniknąć hierarchii klas tworzonych bez potrzeby.

## `@classmethod`

Metoda klasowa otrzymuje klasę jako pierwszy argument:

```python
@classmethod
def method(cls, ...):
    ...
```

Bardzo dobrym zastosowaniem są alternatywne sposoby tworzenia obiektów.

In [ ]:
class TemperatureSensor:

    def __init__(self, name, unit="C"):
        self.name = name
        self.unit = unit

    @classmethod
    def from_dict(cls, data):
        return cls(
            name=data["name"],
            unit=data.get("unit", "C"),
        )


data = {
    "name": "sensor_01",
    "unit": "C",
}

sensor = TemperatureSensor.from_dict(data)

print(sensor.name)
print(sensor.unit)

## Dlaczego `cls`, a nie nazwa klasy?

Zamiast:

```python
return TemperatureSensor(...)
```

piszemy:

```python
return cls(...)
```

Dzięki temu metoda lepiej współpracuje z dziedziczeniem i nie przywiązuje się na sztywno do jednej nazwy klasy.

## `@staticmethod`

Metoda statyczna nie otrzymuje automatycznie ani `self`, ani `cls`.

In [ ]:
class TemperatureSensor:

    @staticmethod
    def celsius_to_kelvin(value):
        return value + 273.15


print(
    TemperatureSensor.celsius_to_kelvin(20)
)

## Czy `staticmethod` zawsze powinien być w klasie?

Nie.

Jeżeli funkcja:

```python
celsius_to_kelvin(value)
```

nie potrzebuje stanu obiektu ani klasy, może być równie dobrze zwykłą funkcją modułu.

`staticmethod` ma sens głównie wtedy, gdy funkcja logicznie należy do interfejsu klasy.

## Klasy przechowujące głównie dane

Czasami klasa ma przede wszystkim reprezentować dane.

Tradycyjnie pisalibyśmy:

In [ ]:
class Measurement:

    def __init__(self, sensor, value, unit):
        self.sensor = sensor
        self.value = value
        self.unit = unit

    def __repr__(self):
        return (
            f"Measurement("
            f"sensor={self.sensor!r}, "
            f"value={self.value!r}, "
            f"unit={self.unit!r})"
        )

## `@dataclass`

Python może wygenerować część powtarzalnego kodu za nas:

In [ ]:
from dataclasses import dataclass


@dataclass
class MeasurementData:
    sensor: str
    value: float
    unit: str


measurement = MeasurementData(
    "sensor_01",
    21.7,
    "C",
)

print(measurement)

## Co otrzymujemy z `dataclass`?

Domyślnie m.in.:

- `__init__`,
- czytelne `__repr__`,
- `__eq__`.

Przykład:

In [ ]:
m1 = MeasurementData("sensor_01", 21.7, "C")
m2 = MeasurementData("sensor_01", 21.7, "C")

print(m1)
print(m1 == m2)

## `dataclass` nie zastępuje wszystkich klas

`dataclass` jest bardzo wygodny dla obiektów, których główną rolą jest przechowywanie danych.

Jeżeli klasa posiada rozbudowane zachowanie, reguły i cykl życia, zwykła klasa nadal może być lepszym wyborem.

## Wyszukiwanie atrybutów

W OOP I widzieliśmy:

```python
sensor.category
```

Python musi ustalić, skąd wziąć wartość `category`.

In [ ]:
class Sensor:

    category = "measurement"

    def __init__(self, name):
        self.name = name


sensor = Sensor("S1")

print(sensor.__dict__)
print(Sensor.__dict__.get("category"))
print(sensor.category)

## Uproszczony model attribute lookup

Dla:

```python
sensor.category
```

możemy na tym etapie myśleć:

```text
instancja
↓
klasa
↓
klasy bazowe
```

Rzeczywisty mechanizm jest bardziej rozbudowany, m.in. przez deskryptory.

## Zasłanianie atrybutu klasy

Jeżeli dodamy:

```python
sensor.category = "special"
```

wartość na instancji może zasłonić wartość klasy.

In [ ]:
sensor.category = "special"

print(sensor.__dict__)
print(sensor.category)
print(Sensor.category)

## Deskryptor

Deskryptor to obiekt, który uczestniczy w obsłudze dostępu do atrybutu.

Może definiować m.in.:

```python
__get__()
__set__()
__delete__()
```

To mechanizm stojący pod wieloma elementami Pythona, w tym `property`.

## Prosty deskryptor walidujący

Przykład demonstracyjny:

In [ ]:
class Positive:

    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self

        return obj.__dict__[self.name]

    def __set__(self, obj, value):
        if value < 0:
            raise ValueError(
                "Value must be non-negative"
            )

        obj.__dict__[self.name] = value


class Tank:

    level = Positive()

    def __init__(self, level):
        self.level = level


tank = Tank(10)
print(tank.level)

## Co daje deskryptor?

Zapis:

```python
tank.level = 10
```

wygląda jak zwykłe przypisanie do atrybutu.

Ale w tle może zostać wykonana dodatkowa logika.

To ta sama ogólna idea, którą poznaliśmy wcześniej przy `property`.

## Czy trzeba samodzielnie pisać deskryptory?

Na tym kursie najważniejsze jest:

> rozumieć, że taki mechanizm istnieje.

W zwykłych programach częściej będziemy korzystać z gotowych mechanizmów takich jak:

- `property`,
- metody,
- biblioteki,
- frameworki.

## Klasy też są obiektami

Sprawdźmy:

In [ ]:
class SensorExample:
    pass

sensor = SensorExample()

print(type(sensor))
print(type(SensorExample))

## Obiekt → klasa → metaklasa

```text
sensor
↓ instance of
SensorExample

SensorExample
↓ instance of
type
```

Czyli sama klasa również jest obiektem.

## Metaklasa

Najprostsza definicja:

> **metaklasa to klasa używana do tworzenia klas.**

Domyślną metaklasą większości klas Pythona jest:

```python
type
```

Na tym etapie ważniejsze jest rozumienie modelu niż samodzielne tworzenie metaklas.

## Dlaczego wspominamy o metaklasach?

Ponieważ pomagają zrozumieć spójność modelu Pythona:

```text
wartość jest obiektem
funkcja jest obiektem
klasa jest obiektem
```

To rozwija model:

```text
nazwa → obiekt
```

który towarzyszy nam od Wykładu 2.

## Jak te mechanizmy się łączą?

```text
OBIEKT
↓
__str__ / __repr__
↓
współpraca z funkcjami języka

OBIEKT
↓
__eq__ / __lt__
↓
współpraca z operatorami

KLASY
↓
dziedziczenie / kompozycja
↓
organizacja zachowania

KLASA
↓
dataclass / descriptor / type
↓
Python Data Model
```

## Przykład integrujący

Połączmy kilka elementów:

In [ ]:
from dataclasses import dataclass


@dataclass
class Measurement:
    sensor: str
    value: float
    unit: str


class Sensor:

    def __init__(self, name):
        self.name = name
        self.value = None

    def update(self, value):
        self.value = value

    def measurement(self, unit):
        return Measurement(
            sensor=self.name,
            value=self.value,
            unit=unit,
        )


class TemperatureSensor(Sensor):

    def __init__(self, name, unit="C"):
        super().__init__(name)
        self.unit = unit

    def describe(self):
        return (
            f"{self.name}: "
            f"{self.value} °{self.unit}"
        )

    @classmethod
    def from_dict(cls, data):
        return cls(
            name=data["name"],
            unit=data.get("unit", "C"),
        )

In [ ]:
sensor = TemperatureSensor.from_dict({
    "name": "sensor_01",
    "unit": "C",
})

sensor.update(21.7)

print(sensor.describe())
print(sensor.measurement(sensor.unit))

## Ćwiczenie

Zaprojektuj system urządzeń:

```text
Device
├── Sensor
└── Actuator
```

Wymagania:

1. `Device` przechowuje `name`,
2. każda klasa ma metodę `describe()`,
3. `Sensor` przechowuje wartość pomiaru,
4. `Actuator` przechowuje stan `enabled`,
5. utwórz listę różnych urządzeń i wywołaj `describe()` bez sprawdzania typu,
6. dodaj `@classmethod from_dict()` do jednej klasy,
7. wybierz jedno miejsce, w którym lepsza będzie kompozycja niż dziedziczenie.

## Podsumowanie

Po tym wykładzie powinieneś pamiętać:

- `__repr__` opisuje techniczną reprezentację obiektu,
- `__str__` opisuje reprezentację czytelną dla użytkownika,
- `__eq__` definiuje znaczenie `==`,
- przy nieobsługiwanym typie porównania warto zwracać `NotImplemented`,
- dziedziczenie modeluje relację między klasą bazową i pochodną,
- `super()` współpracuje z hierarchią dziedziczenia,
- polimorfizm pozwala korzystać ze wspólnego interfejsu,
- duck typing opiera się na zachowaniu, nie tylko typie,
- kompozycja często pasuje do relacji HAS-A,
- `classmethod` może tworzyć alternatywne konstruktory,
- `staticmethod` nie otrzymuje `self` ani `cls`,
- `dataclass` redukuje powtarzalny kod dla klas danych,
- attribute lookup obejmuje instancję, klasę i klasy bazowe,
- deskryptory kontrolują dostęp do atrybutów,
- klasy same są obiektami tworzonymi przez metaklasy.

## Co dalej?

Po dwóch wykładach OOP znamy już:

```text
OOP I
↓
klasa, obiekt, stan, metoda, property

OOP II
↓
data model, dziedziczenie, kompozycja,
polimorfizm, dataclass, deskryptory
```

Następny temat możemy poświęcić iteratorom, generatorom, comprehensions i dekoratorom, czyli kolejnym charakterystycznym mechanizmom Pythona.